# ML Pattern Recognition on GWAS/mQTL Data for Cancer Risk Prediction
### Cotiviti Intern Assessment — Proof of Concept Demo
**Author:** Bhavya Sevak | M.S. Biomedical Informatics, Arizona State University  
**Topic:** Clinical Decision Making and Pattern Recognition in Health Care  
**Date:** June 2026

---

## Overview
This notebook demonstrates an end-to-end ML pipeline that:
1. Simulates a GWAS + mQTL dataset (1,000 samples × 5,000 SNPs)
2. Engineers methylation-inspired interaction features
3. Selects informative SNPs via Chi-squared + LASSO regularization
4. Trains an XGBoost classifier for colorectal cancer risk
5. Outputs per-sample Polygenic Risk Scores (PRS) with calibration

**This is a hackathon POC** — the biology is representative, not clinically validated.

In [ ]:
# ── Step 0: Install dependencies (run once) ───────────────────────────────
# !pip install numpy pandas scikit-learn xgboost matplotlib seaborn --quiet

In [ ]:
# ── Step 1: Imports ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'navy': '#1A1F5E', 'teal': '#00A896', 'amber': '#F59E0B', 'muted': '#64748B'}
print('Libraries loaded ✓')

In [ ]:
# ── Step 2: Simulate GWAS + mQTL Dataset ─────────────────────────────────
# Real GWAS: SNPs encoded as dosage (0, 1, 2) for additive allele model
# We simulate 5,000 SNPs; 50 are 'causal' (linked to cancer risk)

N_SAMPLES  = 1000   # cohort size
N_SNPS     = 5000   # total SNPs genotyped
N_CAUSAL   = 50     # truly causal SNPs (unknown to model)
PREVALENCE = 0.30   # case fraction (~colorectal cancer enriched cohort)

# Allele frequencies (realistic MAF range 0.05–0.45)
mafs = np.random.uniform(0.05, 0.45, N_SNPS)

# Dosage matrix (Hardy-Weinberg equilibrium)
X_raw = np.zeros((N_SAMPLES, N_SNPS), dtype=np.float32)
for j, maf in enumerate(mafs):
    X_raw[:, j] = np.random.choice([0, 1, 2], size=N_SAMPLES,
                                    p=[(1-maf)**2, 2*maf*(1-maf), maf**2])

# Assign causal SNPs with log-odds effect sizes (weak polygenic signal)
causal_idx = np.random.choice(N_SNPS, N_CAUSAL, replace=False)
effect_sizes = np.random.normal(0, 0.4, N_CAUSAL)  # realistic GWAS betas

# Linear predictor + mQTL interaction term (methylation amplifies SNP effect)
log_odds = -1.2  # baseline log-odds
for k, idx in enumerate(causal_idx):
    mQTL_modifier = 1 + 0.3 * np.random.normal(size=N_SAMPLES)  # simulated methylation
    log_odds = log_odds + effect_sizes[k] * X_raw[:, idx] * mQTL_modifier

prob = 1 / (1 + np.exp(-log_odds))
y = (np.random.uniform(size=N_SAMPLES) < prob).astype(int)

# Label SNP columns
snp_names = [f'rs_{i:06d}' for i in range(N_SNPS)]
df = pd.DataFrame(X_raw, columns=snp_names)
df['label'] = y

print(f'Dataset shape: {df.shape}')
print(f'Cases (cancer): {y.sum()} | Controls: {(1-y).sum()} | Prevalence: {y.mean():.1%}')
print(f'Causal SNPs (hidden from model): {N_CAUSAL}')

In [ ]:
# ── Step 3: GWAS Summary Statistics (association test) ───────────────────
# In real GWAS: chi-squared or logistic regression per SNP
# Here: chi-squared approximation for speed (standard in PRS pipelines)

X = df.drop('label', axis=1).values
y_arr = df['label'].values

selector = SelectKBest(chi2, k=500)  # Pre-filter to top 500 SNPs
X_filtered = selector.fit_transform(np.abs(X), y_arr)  # chi2 needs non-negative
selected_mask = selector.get_support()
selected_snps = np.array(snp_names)[selected_mask]

# How many of the truly causal SNPs did we recover?
recovered = set(causal_idx) & set(np.where(selected_mask)[0])
print(f'Chi-squared pre-filter: kept {X_filtered.shape[1]} SNPs')
print(f'Causal SNPs recovered: {len(recovered)} / {N_CAUSAL} ({len(recovered)/N_CAUSAL:.0%})')

In [ ]:
# ── Step 4: LASSO Feature Selection (mQTL-aware sparse regression) ────────
# LASSO logistic regression shrinks non-informative SNP coefficients to zero
# Mimics what mQTL integration does: sparsify the causal signal

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

lasso = LogisticRegressionCV(
    Cs=np.logspace(-3, 1, 20),
    cv=5,
    penalty='l1',
    solver='liblinear',
    max_iter=500,
    random_state=42
)
lasso.fit(X_scaled, y_arr)

lasso_mask = lasso.coef_[0] != 0
X_lasso = X_scaled[:, lasso_mask]
lasso_snps = selected_snps[lasso_mask]

print(f'LASSO selected: {lasso_mask.sum()} SNPs (from 500 → {lasso_mask.sum()})')
print(f'Optimal C (regularization): {lasso.C_[0]:.4f}')

In [ ]:
# ── Step 5: XGBoost Cancer Risk Classifier ────────────────────────────────
# XGBoost handles epistatic interactions between SNPs — key advantage over PRS

clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(1 - y_arr.mean()) / y_arr.mean(),  # handle imbalance
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(clf, X_lasso, y_arr, cv=cv,
                              scoring='roc_auc', n_jobs=-1)

print(f'Cross-validated AUC: {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
print(f'Individual folds: {[f"{s:.3f}" for s in auc_scores]}')

# Final fit for downstream analysis
clf.fit(X_lasso, y_arr)

In [ ]:
# ── Step 6: Polygenic Risk Score Output + Calibration ─────────────────────
# PRS = model's predicted probability, calibrated to output reliable risk scores

calibrated_clf = CalibratedClassifierCV(clf, cv='prefit', method='isotonic')
calibrated_clf.fit(X_lasso, y_arr)

prs = calibrated_clf.predict_proba(X_lasso)[:, 1]

# Stratify into risk tiers (clinical reporting standard)
risk_tiers = pd.cut(prs, bins=[0, 0.25, 0.50, 0.75, 1.0],
                     labels=['Low', 'Intermediate', 'High', 'Very High'])

results_df = pd.DataFrame({
    'Sample_ID': [f'S{i:04d}' for i in range(N_SAMPLES)],
    'PRS': prs.round(4),
    'Risk_Tier': risk_tiers,
    'True_Label': y_arr
})

print('Risk tier distribution:')
print(results_df.groupby('Risk_Tier', observed=False)['True_Label'].agg(['count', 'mean']).rename(columns={'mean': 'case_rate'}))
print('\nSample output (first 5 rows):')
print(results_df.head())

In [ ]:
# ── Step 7: Visualizations ────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('GWAS/mQTL ML Pipeline — Cancer Risk Prediction Results',
             fontsize=16, fontweight='bold', color=COLORS['navy'], y=1.01)

# ── Plot 1: ROC Curve ──
ax1 = axes[0, 0]
fpr, tpr, _ = roc_curve(y_arr, prs)
auc_val = roc_auc_score(y_arr, prs)
ax1.plot(fpr, tpr, color=COLORS['teal'], lw=2.5, label=f'XGBoost PRS (AUC = {auc_val:.3f})')
ax1.plot([0,1], [0,1], 'k--', lw=1, alpha=0.5, label='Random classifier')
ax1.fill_between(fpr, tpr, alpha=0.08, color=COLORS['teal'])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve — Cancer Risk Classifier', fontweight='bold', color=COLORS['navy'])
ax1.legend(fontsize=10)
ax1.set_aspect('equal')

# ── Plot 2: PRS Distribution by Case/Control ──
ax2 = axes[0, 1]
ax2.hist(prs[y_arr==0], bins=40, alpha=0.65, color=COLORS['muted'],
         label='Controls', density=True)
ax2.hist(prs[y_arr==1], bins=40, alpha=0.65, color=COLORS['teal'],
         label='Cases (Cancer)', density=True)
ax2.axvline(0.5, color=COLORS['amber'], linestyle='--', lw=1.5, label='Decision threshold')
ax2.set_xlabel('Polygenic Risk Score (PRS)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('PRS Distribution: Cases vs. Controls', fontweight='bold', color=COLORS['navy'])
ax2.legend(fontsize=10)

# ── Plot 3: Feature Importance (Top 20 SNPs) ──
ax3 = axes[1, 0]
feat_imp = pd.Series(clf.feature_importances_, index=lasso_snps).nlargest(20)
colors_bar = [COLORS['teal'] if i < 5 else COLORS['navy'] for i in range(20)]
feat_imp[::-1].plot(kind='barh', ax=ax3, color=colors_bar[::-1])
ax3.set_xlabel('XGBoost Feature Importance (gain)', fontsize=11)
ax3.set_title('Top 20 Predictive SNPs', fontweight='bold', color=COLORS['navy'])
ax3.tick_params(axis='y', labelsize=8)

# ── Plot 4: Risk Tier Case Rates ──
ax4 = axes[1, 1]
tier_stats = results_df.groupby('Risk_Tier', observed=False)['True_Label'].agg(['count', 'mean'])
tier_stats.columns = ['n', 'case_rate']
bar_colors = [COLORS['muted'], COLORS['teal'], COLORS['navy'], COLORS['amber']]
bars = ax4.bar(tier_stats.index, tier_stats['case_rate'] * 100,
               color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, (_, row) in zip(bars, tier_stats.iterrows()):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'n={row["n"]:.0f}', ha='center', va='bottom', fontsize=10,
             color=COLORS['navy'], fontweight='bold')
ax4.set_xlabel('Risk Tier', fontsize=12)
ax4.set_ylabel('Cancer Case Rate (%)', fontsize=12)
ax4.set_title('Case Rate by PRS Risk Tier', fontweight='bold', color=COLORS['navy'])

plt.tight_layout()
plt.savefig('gwas_ml_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: gwas_ml_results.png')

In [ ]:
# ── Step 8: Summary Report ────────────────────────────────────────────────
print('=' * 60)
print('  GWAS/mQTL ML PIPELINE — COTIVITI POC SUMMARY')
print('=' * 60)
print(f'  Cohort:            {N_SAMPLES} samples × {N_SNPS} SNPs')
print(f'  Cancer prevalence: {y_arr.mean():.1%}')
print(f'  Feature selection: Chi-squared (5,000→500) + LASSO (500→{lasso_mask.sum()})')
print(f'  Final model:       XGBoost (n_estimators=300, max_depth=4)')
print(f'  CV AUC:            {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
print(f'  PRS calibrated:    Isotonic regression (isotonic)')
print(f'  Risk tiers:        Low / Intermediate / High / Very High')
print()
print('  Causal SNP recovery (Chi-sq pre-filter):')
print(f'  {len(recovered)} of {N_CAUSAL} true causal SNPs captured ({len(recovered)/N_CAUSAL:.0%})')
print()
print('  Strategic relevance for Cotiviti:')
print('  → Embed PRS tiers into population health dashboards')
print('  → Flag Very High members for proactive screening outreach')
print('  → Link risk tiers to claims utilization for ROI analysis')
print('=' * 60)

---
## References

- Choi, S. W., Mak, T. S. H., & O'Reilly, P. F. (2020). Tutorial: A guide to performing polygenic risk score analyses. *Nature Protocols*, 15(9), 2759–2772.
- Khera, A. V. et al. (2018). Genome-wide polygenic scores for common diseases. *Nature Genetics*, 50, 1219–1224.
- Reel, P. S. et al. (2021). Using machine learning approaches for multi-omics data analysis. *Biotechnology Advances*, 49, 107739.
- XGBoost documentation: https://xgboost.readthedocs.io
- scikit-learn: https://scikit-learn.org

---
*This POC was developed for the Cotiviti Intern Assessment. Data is fully synthetic.*